# Experiment 1.2b-ii: Polar-factor trajectory proxy analysis in $(Q_t, P_t)$ coordinates

This notebook is the presentation and analysis companion to `run_experiment.py` in the same directory.
It intentionally **does not reimplement the training loop**. Instead, it imports the script and runs the exact script-defined computation, then presents the returned structured results.

## Truthful scope for this first completion pass

- **Toy problem**: a 4-layer deep linear network trained on a fixed random batch with a fixed random target map.
- **What is measured**: per-layer polar decompositions $W_t = Q_t P_t$ and descriptive drift diagnostics built from $Q_t$, $P_t$, and $W_t$.
- **What is *not* measured**: direct coordinates on the full deep-linear gauge orbit or a proof that training is literally confined to a Stiefel manifold.
- **Metric caution**: the raw ratios $\|\Delta Q\|/\|\Delta W\|$ and $\|\Delta P\|/\|\Delta W\|$ can exceed 1 and can become unstable when $\|\Delta W\|$ is very small. The notebook therefore emphasizes aggregate norm ratios $\sum\|\Delta Q\|/\sum\|\Delta W\|$ and $\sum\|\Delta P\|/\sum\|\Delta W\|$.
- **Inference caution**: this default result is a **single-seed, fixed-batch** run.


## 1. Notebook setup and safe import of the counterpart script

The next cell locates `run_experiment.py` without relying on `__file__`, imports it safely, and defines small display helpers for tables and Markdown summaries.


In [ ]:
import html
import importlib.util
import platform
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

try:
    from IPython.display import HTML, Image, Markdown, display
except ImportError:
    HTML = Image = Markdown = None
    def display(obj):
        print(obj)

np.set_printoptions(precision=4, suppress=True)

EXPERIMENT_RELATIVE = Path(
    "experiments/Tier_1_Core_Mechanism_Tests/Exp_1.2_Spectrum_Perturbation_Invariance/"
    "1.2b_Perturbation_Accumulation_Over_Trajectory/"
    "1.2b-ii_Trajectory_in_Polar_Coordinates_Q_t_P_t/run_experiment.py"
)


def locate_script():
    cwd = Path.cwd().resolve()
    search_roots = [cwd, *cwd.parents]

    for root in search_roots:
        candidate = root / EXPERIMENT_RELATIVE
        if candidate.exists():
            return candidate.resolve()

    for root in search_roots:
        candidate = root / "run_experiment.py"
        if candidate.exists() and candidate.parent.name == "1.2b-ii_Trajectory_in_Polar_Coordinates_Q_t_P_t":
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not locate run_experiment.py from the current working directory or its parents."
    )


SCRIPT_PATH = locate_script()
EXPERIMENT_DIR = SCRIPT_PATH.parent
PROJECT_ROOT = next(
    (path for path in [EXPERIMENT_DIR, *EXPERIMENT_DIR.parents] if path.name == "Muon_as_RG_Gauge_Fixing"),
    EXPERIMENT_DIR.parent,
)


def import_module_from_path(module_name, path):
    spec = importlib.util.spec_from_file_location(module_name, path)
    module = importlib.util.module_from_spec(spec)
    if spec.loader is None:
        raise ImportError(f"Could not load module from {path}")
    spec.loader.exec_module(module)
    return module


polar_module = import_module_from_path("exp_1_2b_ii_run_experiment", SCRIPT_PATH)


def format_value(value):
    if isinstance(value, (np.floating, float)):
        value = float(value)
        if np.isnan(value):
            return "NaN"
        abs_value = abs(value)
        if abs_value != 0 and (abs_value < 1e-3 or abs_value >= 1e4):
            return f"{value:.3e}"
        return f"{value:.6f}"
    if isinstance(value, (np.integer, int)):
        return str(int(value))
    if isinstance(value, (list, tuple, np.ndarray)):
        arr = np.asarray(value)
        if arr.ndim == 0:
            return format_value(arr.item())
        if arr.size <= 8:
            return html.escape(np.array2string(arr, precision=4, separator=", "))
        preview = np.array2string(arr[:8], precision=4, separator=", ")
        return html.escape(preview + " ...")
    return html.escape(str(value))



def display_markdown(text):
    if Markdown is None:
        print(text)
    else:
        display(Markdown(text))



def display_table(rows, title=None, columns=None):
    if isinstance(rows, dict):
        rows = [{"field": key, "value": value} for key, value in rows.items()]
    rows = list(rows)

    if title:
        display_markdown(f"#### {title}")

    if not rows:
        display_markdown("_No rows to display._")
        return

    if columns is None:
        columns = list(rows[0].keys())

    header_html = "".join(
        f"<th style='text-align:left;padding:6px;border-bottom:1px solid #999'>{html.escape(str(col))}</th>"
        for col in columns
    )

    body_html = []
    for row in rows:
        cells = "".join(
            f"<td style='padding:6px;vertical-align:top'>{format_value(row.get(col, ''))}</td>"
            for col in columns
        )
        body_html.append(f"<tr>{cells}</tr>")

    table_html = (
        "<table>"
        f"<thead><tr>{header_html}</tr></thead>"
        f"<tbody>{''.join(body_html)}</tbody>"
        "</table>"
    )

    if HTML is None:
        for row in rows:
            print(row)
    else:
        display(HTML(table_html))


## 2. Execute the script-defined experiment

The next cell runs the counterpart script via its import-safe `run_experiment()` function, saving the same figures the script normally writes when executed as a standalone file.


In [ ]:
run_start = time.time()
report = polar_module.run_experiment(make_plots=True, out_dir=EXPERIMENT_DIR, verbose=False)
notebook_wall_time = time.time() - run_start

display_markdown(
    f"**Execution completed.** Imported `{SCRIPT_PATH.name}` from `{EXPERIMENT_DIR}` and obtained a structured results dictionary in `{notebook_wall_time:.2f}` s."
)


## 3. Reproducibility, configuration, and runtime context

This section records the exact script path, runtime, environment, and core hyperparameters used for the current run.


In [ ]:
config = report["config"]
meta = report["metadata"]
problem = report["problem"]
assessment = report["assessment"]

run_rows = [
    {"field": "project_root", "value": str(PROJECT_ROOT)},
    {"field": "experiment_dir", "value": str(EXPERIMENT_DIR)},
    {"field": "script_path", "value": meta["script_path"]},
    {"field": "output_dir", "value": meta["out_dir"]},
    {"field": "python", "value": sys.version.split()[0]},
    {"field": "platform", "value": platform.platform()},
    {"field": "numpy", "value": np.__version__},
    {"field": "notebook_wall_time_sec", "value": notebook_wall_time},
    {"field": "script_report_runtime_sec", "value": meta["runtime_sec"]},
    {"field": "scope", "value": meta["scope"]},
]

display_table(run_rows, title="Execution record")

config_rows = [
    {"parameter": "SEED", "value": config["SEED"]},
    {"parameter": "DIM", "value": config["DIM"]},
    {"parameter": "NUM_LAYERS", "value": config["NUM_LAYERS"]},
    {"parameter": "NUM_STEPS", "value": config["NUM_STEPS"]},
    {"parameter": "BATCH_SIZE", "value": config["BATCH_SIZE"]},
    {"parameter": "LR_MUON", "value": config["LR_MUON"]},
    {"parameter": "LR_SGD_SELECTED", "value": report["learning_rates"]["sgd"]},
    {"parameter": "MOMENTUM", "value": config["MOMENTUM"]},
    {"parameter": "NS_ITERS", "value": config["NS_ITERS"]},
    {"parameter": "REPORT_STEPS", "value": config["REPORT_STEPS"]},
    {"parameter": "LR_CANDIDATES_SGD", "value": config["LR_CANDIDATES_SGD"]},
    {"parameter": "STABILITY_STEPS", "value": config["STABILITY_STEPS"]},
]

display_table(config_rows, title="Configuration used in this run")

display_markdown(
    f"**Interpretation.** The experiment preserves the original toy setup: one deterministic seed (`{config['SEED']}`), one fixed random batch, and the same default dimensions and step counts as the script. SGD's learning rate was selected from the candidate list and Muon's learning rate remained fixed at `{config['LR_MUON']}`. That is useful for continuity with the original script, but it should not be described as a fully fairness-proven hyperparameter match."
)


## 4. Problem-instance diagnostics

We inspect the fixed target spectrum, the dimensionality of the factorization, and the numerical quality of the initial polar decompositions.


In [ ]:
dim = config["DIM"]
layer_dof_rows = [
    {"object": "Per-layer weight matrix W", "degrees_of_freedom": dim * dim},
    {"object": "Orthogonal polar factor Q", "degrees_of_freedom": dim * (dim - 1) // 2},
    {"object": "Symmetric PSD polar factor P", "degrees_of_freedom": dim * (dim + 1) // 2},
]

display_table(layer_dof_rows, title="Per-layer dimensionality summary")

singular_values = np.asarray(problem["target_singular_values"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(np.arange(1, singular_values.size + 1), singular_values, marker="o", linewidth=1.5)
ax.set_title("Target-map singular values")
ax.set_xlabel("Index")
ax.set_ylabel("Singular value (log scale)")
ax.grid(True, alpha=0.3)
plt.show()

spectrum_rows = [
    {"field": "target_condition_number", "value": problem["target_condition_number"]},
    {"field": "largest_singular_value", "value": singular_values[0]},
    {"field": "median_singular_value", "value": np.median(singular_values)},
    {"field": "smallest_singular_value", "value": singular_values[-1]},
    {"field": "initial_loss", "value": problem["initial_loss"]},
]

display_table(spectrum_rows, title="Target-spectrum and initialization diagnostics")
display_table(problem["polar_checks"], title="Initial polar-decomposition checks")

display_markdown(
    "**Interpretation.** The SVD plot makes the fixed target spectrum explicit, and the polar-decomposition checks confirm that the numerical decomposition is working cleanly at initialization (tiny reconstruction and symmetry/orthogonality errors, positive minimum eigenvalues for the initial `P` factors)."
)


## 5. Final optimizer summary

The table below compares the two optimizers using both the more stable aggregate norm ratios and the legacy raw-mean ratios.


In [ ]:
summary_rows = []
for key, label in [("sgd", "SGD+momentum"), ("muon", "Muon+momentum")]:
    summary = report["optimizers"][key]["summary"]
    summary_rows.append(
        {
            "optimizer": label,
            "lr": report["optimizers"][key]["lr"],
            "final_loss": summary["final_loss"],
            "aggregate_dQ_over_dW": summary["aggregate_dQ_over_dW"],
            "aggregate_dP_over_dW": summary["aggregate_dP_over_dW"],
            "raw_mean_dQ_ratio": summary["raw_mean_dQ_ratio"],
            "raw_mean_dP_ratio": summary["raw_mean_dP_ratio"],
            "mean_cum_Q_final": summary["mean_cum_Q_final"],
            "mean_cum_P_final": summary["mean_cum_P_final"],
            "mean_cum_Q_fraction_final": summary["mean_cum_Q_fraction_final"],
        }
    )

display_table(summary_rows, title="Final summary metrics")

sgd_summary = report["optimizers"]["sgd"]["summary"]
muon_summary = report["optimizers"]["muon"]["summary"]

display_markdown(
    f"**Interpretation.** Main per-step comparison in this notebook uses aggregate ratios across all layers: `Σ||ΔQ||/Σ||ΔW||` and `Σ||ΔP||/Σ||ΔW||`. Aggregate Q-motion: SGD = `{sgd_summary['aggregate_dQ_over_dW']:.6f}`, Muon = `{muon_summary['aggregate_dQ_over_dW']:.6f}`; therefore the stronger expectation that Muon has larger per-step Q motion than SGD is **{'supported' if assessment['robust_tests']['R1']['passed'] else 'not supported'}** in this run. Aggregate P-motion: SGD = `{sgd_summary['aggregate_dP_over_dW']:.6f}`, Muon = `{muon_summary['aggregate_dP_over_dW']:.6f}`; therefore the expectation that Muon suppresses P-motion relative to SGD is **{'supported' if assessment['robust_tests']['R2']['passed'] else 'not supported'}**. Final cumulative orientation fraction: SGD = `{sgd_summary['mean_cum_Q_fraction_final']:.4f}`, Muon = `{muon_summary['mean_cum_Q_fraction_final']:.4f}`. The raw means are preserved for continuity with the original script, but they are supplementary rather than primary because they are sensitive to tiny `||ΔW||` denominators."
)


## 6. Windowed per-step ratio summaries

These tables retain the original report-step structure, but the first table uses the more stable aggregate norm ratios as the primary summary and the raw means only as a supplemental legacy view.


In [ ]:
window_rows = []
for row in report["tables"]["windowed_ratios"]:
    window_rows.append(
        {
            "step": row["step"],
            "window": f"{row['window'][0]}-{row['window'][1]}",
            "SGD agg dQ/dW": row["sgd_aggregate_dQ_over_dW"],
            "SGD agg dP/dW": row["sgd_aggregate_dP_over_dW"],
            "Muon agg dQ/dW": row["muon_aggregate_dQ_over_dW"],
            "Muon agg dP/dW": row["muon_aggregate_dP_over_dW"],
            "SGD raw dQ/dW": row["sgd_raw_mean_dQ_ratio"],
            "Muon raw dQ/dW": row["muon_raw_mean_dQ_ratio"],
        }
    )

display_table(window_rows, title="Windowed per-step ratio summaries")

display_markdown(
    f"**Interpretation.** Across the report windows, the aggregate ratios show how much Q- and P-factor motion accumulates relative to total weight motion. The raw means are kept only for continuity with the older narrative. In the present run, the robust test for larger Muon Q-motion is **{'passed' if assessment['robust_tests']['R1']['passed'] else 'not passed'}**, while the robust test for smaller Muon P-motion is **{'passed' if assessment['robust_tests']['R2']['passed'] else 'not passed'}**."
)


## 7. Cumulative drift views and saved figures

The cumulative tables and figures show how far the layerwise polar factors move away from their initial values over training.


In [ ]:
cumulative_rows = []
for row in report["tables"]["cumulative_drift"]:
    cumulative_rows.append(
        {
            "step": row["step"],
            "SGD ||Q-Q0||": row["sgd_cum_Q"],
            "SGD ||P-P0||": row["sgd_cum_P"],
            "SGD Q/(Q+P)": row["sgd_Q_fraction"],
            "Muon ||Q-Q0||": row["muon_cum_Q"],
            "Muon ||P-P0||": row["muon_cum_P"],
            "Muon Q/(Q+P)": row["muon_Q_fraction"],
        }
    )

display_table(cumulative_rows, title="Cumulative drift summaries at report steps")

for key, caption in [
    ("overview", "Overview figure produced by the script"),
    ("phase_portrait", "Phase portrait produced by the script"),
]:
    path = report["plots"].get(key)
    if path and Path(path).exists() and Image is not None:
        display_markdown(f"#### {caption}")
        display(Image(filename=str(path)))
    elif path:
        display_markdown(f"Saved figure: `{path}`")

sgd_q_frac = sgd_summary["mean_cum_Q_fraction_final"]
muon_q_frac = muon_summary["mean_cum_Q_fraction_final"]

display_markdown(
    f"**Interpretation.** By the final step, the layer-averaged cumulative orientation fraction is `{sgd_q_frac:.4f}` for SGD and `{muon_q_frac:.4f}` for Muon. That is a modest cumulative difference in favor of Muon. It is informative as a proxy diagnostic, but it should not be overstated as direct evidence that the network has been resolved into true gauge versus physical coordinates."
)


## 8. Per-layer variation, early/late behavior, and ratio diagnostics

To keep the notebook academically serious, we inspect per-layer variation and explicitly show why the raw ratio metrics should be treated with caution.


In [ ]:
display_table(report["tables"]["per_layer_final"]["sgd"], title="Per-layer final summaries: SGD")
display_table(report["tables"]["per_layer_final"]["muon"], title="Per-layer final summaries: Muon")

layers = [row["layer"] for row in report["tables"]["per_layer_final"]["sgd"]]
sgd_q_fraction_by_layer = [row["Q_fraction_final"] for row in report["tables"]["per_layer_final"]["sgd"]]
muon_q_fraction_by_layer = [row["Q_fraction_final"] for row in report["tables"]["per_layer_final"]["muon"]]

x = np.arange(len(layers))
width = 0.35
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(x - width / 2, sgd_q_fraction_by_layer, width=width, label="SGD")
ax.bar(x + width / 2, muon_q_fraction_by_layer, width=width, label="Muon")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1, alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels(layers)
ax.set_xlabel("Layer")
ax.set_ylabel("Final cumulative Q/(Q+P)")
ax.set_title("Per-layer final cumulative orientation fraction")
ax.legend()
ax.grid(True, axis="y", alpha=0.3)
plt.show()

phase_rows = []
for row in report["tables"]["phase_comparison"]:
    phase_rows.append(
        {
            "phase": row["phase"],
            "SGD agg dQ/dW": row["sgd_aggregate_dQ_over_dW"],
            "SGD agg dP/dW": row["sgd_aggregate_dP_over_dW"],
            "Muon agg dQ/dW": row["muon_aggregate_dQ_over_dW"],
            "Muon agg dP/dW": row["muon_aggregate_dP_over_dW"],
            "SGD raw dQ/dW": row["sgd_raw_mean_dQ_ratio"],
            "Muon raw dQ/dW": row["muon_raw_mean_dQ_ratio"],
        }
    )

display_table(phase_rows, title="Early-vs-late training comparison")

diagnostic_rows = []
for key, label in [("sgd", "SGD"), ("muon", "Muon")]:
    summary = report["optimizers"][key]["summary"]
    diagnostic_rows.append(
        {
            "optimizer": label,
            "small_dW_count": summary["small_dW_count"],
            "small_dW_fraction": summary["small_dW_fraction"],
            "dW_min": summary["dW_stats"]["min"],
            "dW_median": summary["dW_stats"]["median"],
            "raw_dQ_ratio_max": summary["dQ_ratio_stats"]["max"],
            "raw_dP_ratio_max": summary["dP_ratio_stats"]["max"],
        }
    )

display_table(diagnostic_rows, title="Why the raw ratio means are only supplemental")

display_markdown(
    "**Interpretation.** The per-layer tables and bar plot show that the cumulative orientation fraction is not identical across layers, so any blanket statement about the optimizer should be read as a layer-averaged tendency rather than an exact per-layer law. The diagnostic table also makes the raw-ratio caveat concrete: large maxima can occur because some steps have very small `||ΔW||`, which is why the aggregate norm ratios are emphasized throughout this notebook."
)


## 9. Test outcomes and calibrated conclusion

The final section reports both the legacy continuity tests retained from the original script and the more cautious calibrated conclusion now returned by the refactored script.


In [ ]:
legacy_rows = []
for test_id, payload in report["assessment"]["legacy_tests"].items():
    legacy_rows.append(
        {
            "test": test_id,
            "description": payload["description"],
            "Muon": payload["muon"],
            "SGD": payload["sgd"],
            "passed": payload["passed"],
        }
    )

robust_rows = []
for test_id, payload in report["assessment"]["robust_tests"].items():
    robust_rows.append(
        {
            "test": test_id,
            "description": payload["description"],
            "Muon": payload["muon"],
            "SGD": payload["sgd"],
            "passed": payload["passed"],
        }
    )

display_table(legacy_rows, title="Legacy continuity tests")
display_table(robust_rows, title="Robust supplemental tests")

conclusion = report["assessment"]["calibrated_conclusion"]
supported_block = "\n".join(f"- {item}" for item in conclusion["supported_expectations"])
unsupported_block = "\n".join(f"- {item}" for item in conclusion["unsupported_expectations"])
limitations_block = "\n".join(f"- {item}" for item in conclusion["limitations"])

display_markdown(f"""## Calibrated conclusion

**Label:** {conclusion['label']}

{conclusion['detail']}

**What was supported in the current run**
{supported_block}

**What was not supported in the current run**
{unsupported_block}

**Important limitations**
{limitations_block}

**Bottom line.** This notebook now matches the script-defined computation and presents it as a **per-layer polar-factor proxy analysis**. In the current single-seed toy run, Muon achieves a lower final loss, lower P-directed motion, and a somewhat higher cumulative orientation fraction than SGD, but stronger claims about larger per-step Q motion or direct measurement of deep-linear gauge fixing are not warranted from these metrics alone.
""")
